# Key Exploratory Findings

## 1. Indicator Overview and Baseline

The **Minimum Dietary Diversity for Women (MDD-W)** is a population-level indicator used to assess dietary diversity among non-pregnant women aged 15–49 years. It measures whether a woman consumed foods from at least **5 out of 10 defined food groups** during the previous 24 hours. Achieving this threshold is associated with a higher likelihood of adequate micronutrient intake.

To prevent very small quantities of food from artificially improving dietary diversity scores, foods consumed in amounts below **15 grams (approximately one tablespoon)** are excluded from the assessment.

---

## 2. Global Performance Spectrum

### High-Performing Countries

The countries with the highest national rates of minimum dietary diversity are:

* **Tajikistan:** 80.4%
* **Tunisia:** 80.2%
* **Jordan:** 76.4%

These countries demonstrate the highest proportions of women meeting the MDD-W threshold among the countries included in the dataset.

### Low-Performing Countries

The lowest national dietary diversity rates were observed in:

* **Uganda:** 12.7%
* **Ethiopia:** 16.6%
* **Lesotho:** 18.2%

The substantial gap between the highest- and lowest-performing countries highlights significant differences in women's access to diverse diets across regions and countries.

---

## 3. The Rural-Urban Divide

Across nearly all of the **25 countries** covered in the dataset, women living in urban areas recorded higher dietary diversity rates than those living in rural areas.

However, **Brazil (2014)** presents a notable exception to this overall pattern:

* **Rural women:** 70.1%
* **Urban women:** 54.4%

This reverse rural-urban pattern suggests that access to diverse diets may be influenced by country-specific factors such as agricultural production, food availability, cultural dietary practices, and urban food environments.

---

## 4. West African Dietary Patterns

A closer examination of the six West African countries included in the dataset, **Burkina Faso, Côte d'Ivoire, Ghana, Nigeria, Senegal, and Sierra Leone**, reveals several notable dietary patterns.

### Staple Food Dominance

Consumption of **grains, white roots, and tubers** was almost universal across the six countries, ranging from **94.5% to 98.9%**.

This suggests that staple foods form the foundation of women's diets across the region.

### Low Consumption of Animal-Source Foods

Egg consumption was consistently low across the six countries, ranging from:

* **6.7% in Burkina Faso**
* to **27.6% in Ghana**

Dairy consumption was also generally low, with **Senegal standing out as an exception at 48.3%**.

The relatively low consumption of eggs and dairy may point to potential gaps in access to important sources of protein and micronutrients.

### The Sweet Beverage Pattern

Sweet beverage consumption emerged as another notable dietary feature in the region.

The most striking case was **Senegal**, where **89.5% of women reported consuming sweet beverages** during the 24-hour recall period.

This finding highlights an important distinction between simply consuming foods from multiple groups and the overall nutritional quality of those foods.

---

## 5. Key Methodological Limitations

Several limitations should be considered when interpreting the findings:

* **Limited time-series data:** Most countries have only one survey observation, making it difficult to analyse trends or changes in dietary diversity over time.

* **Recall bias:** The data relies on respondents accurately remembering everything consumed during the previous 24 hours.

* **Social desirability bias:** Respondents may over-report foods perceived as healthy or under-report foods perceived as unhealthy.

* **Limited information on quantity and nutrient quality:** The 24-hour recall approach does not provide detailed information on portion sizes, nutrient density, or absolute nutrient intake.

These limitations mean that the MDD-W indicator should be interpreted primarily as a measure of **dietary diversity and the likelihood of micronutrient adequacy**, rather than a complete assessment of an individual's overall nutritional intake.


In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [ ]:


# Load the FAOSTAT dataset from your local path
df = pd.read_csv('/content/Minimum_Dietary_Diversity_for_Women_(MDD-W)_Food_and_Diet_E_All_Data_(Normalized).csv')

# Define West African surveys and the 10 core MDD-W food groups
west_africa_surveys = [
    'Burkina Faso - 2021',
    "Côte d'Ivoire - 2021",
    'Ghana - 2022',
    'Nigeria - 2023',
    'Senegal - 2023',
    'Sierra Leone - 2019'
]

country_map = {
    'Burkina Faso - 2021': 'Burkina Faso (2021)',
    "Côte d'Ivoire - 2021": "Côte d'Ivoire (2021)",
    'Ghana - 2022': 'Ghana (2022)',
    'Nigeria - 2023': 'Nigeria (2023)',
    'Senegal - 2023': 'Senegal (2023)',
    'Sierra Leone - 2019': 'Sierra Leone (2019)'
}

core_food_groups = [
    'Grains, white roots and tubers, and plantains',
    'Pulses (beans, peas and lentils)',
    'Nuts and seeds',
    'Dairy',
    'Meat, poultry and fish',
    'Eggs',
    'Dark green leafy vegetables',
    'Other vitamin A-rich fruits and vegetables',
    'Vegetables, other',
    'Fruits, other'
]

# Filter dataset to extract national-level food group consumption rates (Indicator 6212)
df_filtered = df[
    (df['Survey'].isin(west_africa_surveys)) &
    (df['Geographic Level'] == 'National') &
    (df['Indicator Code'] == 6212) &
    (df['Food Group'].isin(core_food_groups))
].copy()

# Pivot the long-format data into wide-format
pivoted = df_filtered.pivot(index='Survey', columns='Food Group', values='Value')
pivoted = pivoted[core_food_groups]  # Reorder columns logically

# Calculate total heights for sorting/annotation placement
pivoted['Total'] = pivoted.sum(axis=1)

# Hardcode the official MDD-W population achievement rates (Indicator 6211) for bar annotations
mddw_rates = {
    'Senegal - 2023': 62.8,
    'Sierra Leone - 2019': 56.4,
    'Ghana - 2022': 49.9,
    'Nigeria - 2023': 43.2,
    "Côte d'Ivoire - 2021": 29.4,
    'Burkina Faso - 2021': 25.1
}
pivoted['MDD-W Rate'] = pivoted.index.map(mddw_rates)

# Sort descending by population-level dietary adequacy (MDD-W Rate)
pivoted_sorted = pivoted.sort_values(by='MDD-W Rate', ascending=False)
pivoted_sorted.index = pivoted_sorted.index.map(country_map)

# Melt back to long format for Plotly Express (it wants tidy data, not a wide pivot)
plot_df = pivoted_sorted[core_food_groups].reset_index().melt(
    id_vars='Survey', var_name='Food Group', value_name='Value'
)
plot_df = plot_df.rename(columns={'Survey': 'Country'})

# Preserve the sorted country order and stacking order as categorical axes
country_order = list(pivoted_sorted.index)
plot_df['Country'] = pd.Categorical(plot_df['Country'], categories=country_order, ordered=True)
plot_df['Food Group'] = pd.Categorical(plot_df['Food Group'], categories=core_food_groups, ordered=True)
plot_df = plot_df.sort_values(['Country', 'Food Group'])

# Colorblind-safe palette (matches seaborn's 'colorblind' palette used previously)
colorblind_palette = [
    '#0173B2', '#DE8F05', '#029E73', '#D55E00', '#CC78BC',
    '#CA9161', '#FBAFE4', '#949494', '#ECE133', '#56B4E9'
]

fig = px.bar(
    plot_df,
    x='Country',
    y='Value',
    color='Food Group',
    color_discrete_sequence=colorblind_palette,
    category_orders={'Country': country_order, 'Food Group': core_food_groups},
    labels={
        'Value': 'Sum of Food Group Consumption Rates (% points)',
        'Country': 'Country & Survey Year'
    },
    title="Animal-Source Deficits Constrain West African Diets: Eggs and Dairy Largely Absent"
)

# Reverse legend order to match visual stack order (top-to-bottom), as in the original
fig.update_layout(legend=dict(traceorder='reversed', title='10 Core Food Groups'))

fig.update_layout(
    barmode='stack',
    yaxis=dict(range=[0, 600], title='Sum of Food Group Consumption Rates (% points)<br>[Height = Mean Food Group Score x 100]'),
    xaxis_title='Country & Survey Year',
    title_font=dict(size=18),
    template='plotly_white',
    width=1100,
    height=750,
    margin=dict(t=90, b=90, r=220)
)

# Annotate each bar with the percentage of women actually achieving the MDD-W score
for country in country_order:
    total = pivoted_sorted.loc[country, 'Total']
    rate = pivoted_sorted.loc[country, 'MDD-W Rate']
    fig.add_annotation(
        x=country,
        y=total + 15,
        text=f"MDD-W:<br>{rate:.1f}%",
        showarrow=False,
        font=dict(size=12, color='#1A3333', family='Arial Black'),
        align='center'
    )

# Add source footnote
fig.add_annotation(
    text='Source: FAOSTAT Minimum Dietary Diversity for Women (MDD-W) Database (2024)',
    xref='paper', yref='paper',
    x=0, y=-0.16,
    showarrow=False,
    font=dict(size=10, color='gray'),
    align='left'
)



In [ ]:
# Save figure (static image requires the 'kaleido' package: pip install -U kaleido)
os.makedirs('/workspace/scratch', exist_ok=True)
fig.write_image('/workspace/scratch/west_africa_mddw_stacked.png', scale=2)

# Optionally also save an interactive HTML version
fig.write_html('/workspace/scratch/west_africa_mddw_stacked.html')

## Global Urban vs. Rural Dietary Diversity Gaps

In [ ]:
import os
import pandas as pd
import plotly.graph_objects as go

# LOAD DATA: Same CSV path verification
data_path = '/content/Minimum_Dietary_Diversity_for_Women_(MDD-W)_Food_and_Diet_E_All_Data_(Normalized).csv'

if not os.path.exists(data_path):
    print(f"ERROR: File not found.")
else:
    df = pd.read_csv(data_path)

    # Filter for the population dietary diversity metric (Indicator 6211)
    df_mddw = df[df['Indicator Code'] == 6211].copy()

    # Separate urban, rural and national profiles, then pivot wide
    piv_geo = df_mddw.pivot(index='Survey', columns='Geographic Level', values='Value')
    piv_geo = piv_geo.dropna(subset=['Urban', 'Rural'])  # Exclude rows without spatial splits

    # Calculate gap
    piv_geo['Gap'] = piv_geo['Urban'] - piv_geo['Rural']
    piv_geo_sorted = piv_geo.sort_values(by='Gap', ascending=False)

    # Colorblind-safe colors, matching seaborn's 'colorblind' palette positions 0 and 7
    urban_color = '#0173B2'
    rural_color = '#949494'

    fig = go.Figure()

    # Draw the connecting line for each row (the "gap")
    for survey, row in piv_geo_sorted.iterrows():
        fig.add_trace(go.Scatter(
            x=[row['Rural'], row['Urban']],
            y=[survey, survey],
            mode='lines',
            line=dict(color='lightgray', width=3),
            showlegend=False,
            hoverinfo='skip'
        ))

    # Urban points
    fig.add_trace(go.Scatter(
        x=piv_geo_sorted['Urban'],
        y=piv_geo_sorted.index,
        mode='markers',
        name='Urban',
        marker=dict(color=urban_color, size=12),
    ))

    # Rural points
    fig.add_trace(go.Scatter(
        x=piv_geo_sorted['Rural'],
        y=piv_geo_sorted.index,
        mode='markers',
        name='Rural',
        marker=dict(color=rural_color, size=12),
    ))

    # Gap annotations
    for survey, row in piv_geo_sorted.iterrows():
        x_pos = max(row['Urban'], row['Rural']) + 1.5
        fig.add_annotation(
            x=x_pos,
            y=survey,
            text=f"Gap: {row['Gap']:.1f} pp",
            showarrow=False,
            xanchor='left',
            font=dict(size=11, color='gray', family='Arial Black'),
        )

    # Formatting — keep the same category order top-to-bottom as the sorted index
    fig.update_layout(
        title=dict(
            text="Tanzania & Cambodia Exhibit Widest Urban-Rural Gaps, While Brazil Inverts the Trend",
            font=dict(size=15, family='Arial Black')
        ),
        xaxis_title='Percentage (%) of Women Achieving Minimum Dietary Diversity (MDD-W)',
        yaxis_title='Country Survey and Year',
        yaxis=dict(
            categoryorder='array',
            categoryarray=list(piv_geo_sorted.index)[::-1],  # reversed so highest gap sits at top
        ),
        template='plotly_white',
        legend=dict(x=0.85, y=0.02, xanchor='left', yanchor='bottom', bordercolor='lightgray', borderwidth=1),
        width=900,
        height=900,
        margin=dict(l=200, r=150, t=80, b=60),
    )

    fig.show()

    # Optional: save outputs
    # fig.write_image('/content/urban_rural_gap.png', scale=2)  # requires: pip install -U kaleido
    # fig.write_html('/content/urban_rural_gap.html')